# 22 — Cleaning, candidate pairs and features (Milestone 3)

Owner: Member 3. Milestone 3 matches every BibTeX entry of a publication (M2 `bibtex` table) to the
arXiv-keyed references of the **same** publication (M1 `references` table). For a publication with `m`
entries and `n` references the candidates form an `m × n` matrix; each labelled entry has at most one
positive cell. We treat it as **learning to rank**: for each entry, the `n` references are ranked and the
top 5 are scored with MRR@5 (`23_model`).

This notebook only defines functions; tests and the demonstration on the dev sample run under `RUN_TEST`.

| Section | Functions |
|---|---|
| 1. Cleaning | `author_surname`, `clean_entries`, `clean_refs`, `cleaning_report` |
| 2. Pairs and labels | `build_pairs`, `read_label_files`, `read_split`, `attach_labels` |
| 3. Features | `fit_text_models`, `add_features`, `FEATURES`, `FEATURE_DOC`, `feature_table_markdown` |
| 4. Ranking metric | `rank_candidates`, `gold_of`, `mrr_at_k`, `hit_at_k` (used again by `23_model`) |
| 5. Feature analysis | `feature_report`, `plot_feature_histograms`, `feature_findings` |
| 6. Full run | `run_features` (called by `25_run_m3`) |
| 7. Dev sample | `dev_frames`, `dev_labels` (only to try the code before the real labels exist) |
| 8. Tests | `FeatureTests` |

**Inputs** (data contract of `context/plan-m3.md`, Drive `MyDrive/arxiv-project/`):
`data/m3/entries.parquet`, `data/m3/refs.parquet` (from `20_m3_data`), `labels/manual/*.csv`,
`labels/auto/auto_labels.parquet`, `labels/split.json` (from `21_labeling`).
**Outputs:** `data/m3/pairs.parquet`, `logs/m3_feature_report.csv`, `logs/m3_cleaning_report.csv`,
`logs/figures/m3/feat_*.png`.

Only classical methods: TF-IDF vectorizers are fitted on the titles of our own corpus (unsupervised, no
labels, no pretrained model); string similarities come from `rapidfuzz`.

In [ ]:
# Helpers of 12_bibtex and 03_references: from 20_m3_data when it exists, else loaded here the same way.
from pathlib import Path

if "RUN_TEST" not in globals():
    RUN_TEST = True
_RUN_TEST_22 = RUN_TEST
if "H12" not in globals() or "H03" not in globals():
    RUN_TEST = False
    if Path("./20_m3_data.ipynb").is_file():
        get_ipython().run_line_magic("run", "-i ./20_m3_data.ipynb")
    if "H12" not in globals() or "H03" not in globals():
        if "load_definitions" not in globals():
            get_ipython().run_line_magic("run", "-i ./13_dedup_export.ipynb")   # also runs 00_config
        H12 = TEAM["12_bibtex"] if "12_bibtex" in TEAM else load_definitions("12_bibtex", {**globals()})
        # Only normalize_ref_arxiv_id is needed from 03 (the rest of 03 sets up the S2 client); its helpers
        # (split_version, NEW_ID_RE, OLD_ID_RE, to_folder_id) come from 00_config.
        import ast as _ast
        H03 = {**globals()}
        for _cell in json.loads(Path("./03_references.ipynb").read_text(encoding="utf-8"))["cells"]:
            if _cell["cell_type"] == "code" and "def normalize_ref_arxiv_id" in "".join(_cell["source"]):
                for _node in _ast.parse("".join(_cell["source"])).body:
                    if isinstance(_node, _ast.FunctionDef) and _node.name == "normalize_ref_arxiv_id":
                        exec(compile(_ast.Module([_node], []), "<03_references>", "exec"), H03)
RUN_TEST = _RUN_TEST_22

import json
import math
import re
import time
import warnings

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from scipy import sparse
from scipy.stats import rankdata
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.metrics import roc_auc_score

SEED = 42
M3_DIR = globals().get("M3_DIR", DATA_DIR.parent / "m3")          # data/m3/
LABEL_DIR = globals().get("LABEL_DIR", PROJECT_ROOT / "labels")    # labels/manual, labels/auto, split.json
M3_FIG_DIR = LOG_DIR / "figures" / "m3"
PAIRS_PATH = M3_DIR / "pairs.parquet"
FEATURE_REPORT_PATH = LOG_DIR / "m3_feature_report.csv"
CLEANING_REPORT_PATH = LOG_DIR / "m3_cleaning_report.csv"
KEY = ["arxiv_id", "citation_key"]          # one entry
CELL = KEY + ["ref_arxiv_id"]               # one candidate pair
print(f"22 loaded | M3_DIR: {M3_DIR} | labels: {LABEL_DIR}")

## 1. Cleaning

The statement asks for lowercasing, stop words and tokenization of titles and author names. Both sides of
a pair are cleaned by the **same** functions, so that a difference in a feature comes from the data and
not from two normalizations.

- **Titles**: `norm_title` of 12 (LaTeX to text, accents folded, lower case, `[a-z0-9]` tokens). Tokens
  without the English stop words of scikit-learn feed the word features; numbers are kept (`gpt 4`,
  `3d`). The character TF-IDF uses the full cleaned title.
- **Authors**: one key per author, the **last word of the family name**, folded to `[a-z]`:
  `A. B. Smith`, `Smith, A. B.` and `Smith Jr., A.` give `smith`; `Johannes van der Waals` and
  `van der Waals, J. D.` give `waals` (S2 writes particles in either place, so the last word is the
  part both sides agree on). `others`, `et al.` and `and others` are dropped, suffixes (`Jr.`, `II`) too.
- **Identifiers**: `arxiv_hint` is `find_arxiv_id` of 12 on `doi`, `url` and `journal` (an arXiv URL,
  `arXiv:2310.06825`, a `10.48550/arXiv.*` DOI), normalized with `normalize_ref_arxiv_id` of 03, so old
  ids like `hep-th/9901001` keep their form. `doi_norm` is `norm_doi` of the `doi` field (or the URL).
- **Year**: the `year` column of the entry; for a reference, the S2 year, else the first 4 characters of
  `ref_submission_date`.
- **`raw_text`**: title, author keys and venue together, a fallback for `\bibitem` entries whose title
  could not be separated from the rest of the item.

In [ ]:
_NAME_SUFFIXES = {"jr", "sr", "jun", "junior", "ii", "iii", "iv"}
_NOT_AUTHORS = {"others", "et al", "and others", "al", "et al.", ""}
_ET_AL_RE = re.compile(r"(?:\band\s+others|\bet\s*\.?\s*al\.?)\s*$", re.I)
_CITE_CMD_RE = re.compile(r"\\[A-Za-z]*cite[A-Za-z]*\*?(?:\[[^\]]*\])*\{[^}]*\}")
STOP_WORDS = frozenset(ENGLISH_STOP_WORDS)


def _text(value):
    """None / NaN / NA -> ''; anything else -> str."""
    if value is None or (isinstance(value, float) and math.isnan(value)) or value is pd.NA:
        return ""
    return str(value)


def _as_list(value):
    """Authors cell (list, numpy array, None, or one string) -> list of str."""
    if value is None or value is pd.NA or (isinstance(value, float) and math.isnan(value)):
        return []
    if isinstance(value, str):
        return [value] if value.strip() else []
    return [str(v) for v in list(value) if v is not None and str(v).strip()]


def _is_suffix(word):
    return H12["_ascii_fold"](word).lower().strip(" .") in _NAME_SUFFIXES


def author_surname(name):
    """'A. B. Smith', 'Smith, A. B.', 'Smith Jr., A.' -> 'smith'; 'van der Waals, J.' -> 'waals'; 'others' -> ''."""
    s = H12["latex_to_text"](_text(name)).strip()
    s = _ET_AL_RE.sub("", s).strip(" ,;")
    if s.lower().strip(" .") in _NOT_AUTHORS:
        return ""
    if "," in s:                                     # 'Last, First' / 'Last, Jr., First' / 'Last Jr., First'
        parts = [p.strip() for p in s.split(",") if p.strip() and not _is_suffix(p)]
        family = parts[0] if parts else ""
    else:
        family = s
    words = [w for w in family.split() if not _is_suffix(w)]
    return H12["surname_key"](words[-1]) if words else ""


def author_surnames(authors):
    """List of author keys, empty ones dropped, order kept."""
    return [k for k in (author_surname(a) for a in _as_list(authors)) if k]


def title_tokens(clean_title):
    """Content tokens of a cleaned title: English stop words removed, numbers kept."""
    return [t for t in clean_title.split() if t not in STOP_WORDS]


def _arxiv_id(raw):
    """Any arXiv id spelling -> canonical id without version ('' if it is not one)."""
    return H03["normalize_ref_arxiv_id"](raw)[0] or "" if raw else ""


def _year(value):
    """Int year or pd.NA."""
    if value is None or value is pd.NA:
        return pd.NA
    try:
        if isinstance(value, float) and math.isnan(value):
            return pd.NA
        y = int(value)
        return y if 1000 <= y <= 2100 else pd.NA
    except (TypeError, ValueError):
        y = H12["parse_year"](value)
        return y if y else pd.NA


def _context_text(value):
    """Citing sentences (str or list) without the \\cite commands, normalized like titles."""
    if isinstance(value, (list, tuple, np.ndarray)):
        value = " ".join(_text(v) for v in value)
    return H12["norm_title"](_CITE_CMD_RE.sub(" ", _text(value)))


def clean_entries(entries):
    """Contract `entries` (9 bibtex columns + optional bib_source, position, n_cites, context) -> + clean columns."""
    e = entries.copy().reset_index(drop=True)
    for column, default in (("bib_source", "unknown"), ("n_cites", np.nan), ("context", ""), ("position", None)):
        if column not in e.columns:
            e[column] = default
    if e["position"].isna().all():                   # bibliography order = row order within the paper
        e["position"] = e.groupby("arxiv_id", sort=False).cumcount()
    e["arxiv_id"] = e["arxiv_id"].astype(str)
    e["title_clean"] = [H12["norm_title"](_text(t)) for t in e["title"]]
    e["title_tokens"] = [title_tokens(t) for t in e["title_clean"]]
    e["surnames"] = [author_surnames(a) for a in e["authors"]]
    e["first_surname"] = [s[0] if s else "" for s in e["surnames"]]
    e["n_authors"] = [len(s) for s in e["surnames"]]
    e["year"] = pd.array([_year(y) for y in e["year"]], dtype="Int64")
    fields = zip(e["doi"].map(_text), e["url"].map(_text), e["journal"].map(_text))
    e["arxiv_hint"] = [_arxiv_id(H12["find_arxiv_id"]({"doi": d, "url": u, "journal": j})) for d, u, j in fields]
    e["doi_norm"] = [H12["norm_doi"](_text(d)) or H12["norm_doi"](_text(u)) for d, u in zip(e["doi"], e["url"])]
    e["venue_clean"] = [H12["norm_title"](_text(j)) for j in e["journal"]]
    e["is_arxiv_venue"] = [("arxiv" in v.split()) or ("arxiv.org" in _text(u).lower()) or d.startswith("10.48550/")
                           for v, u, d in zip(e["venue_clean"], e["url"], e["doi_norm"])]
    e["raw_text"] = [" ".join(filter(None, (t, " ".join(s), v)))
                     for t, s, v in zip(e["title_clean"], e["surnames"], e["venue_clean"])]
    e["context_clean"] = [_context_text(c) for c in e["context"]]
    return e


def clean_refs(refs):
    """Contract `refs` (6 references columns + ref_year, ref_doi, ref_venue, ref_position) -> + clean columns.

    A reference listed twice for one paper (S2 sometimes does) is kept once: the first one.
    """
    r = refs.copy().reset_index(drop=True)
    for column in ("ref_year", "ref_doi", "ref_venue", "ref_position", "ref_submission_date"):
        if column not in r.columns:
            r[column] = None
    r["arxiv_id"] = r["arxiv_id"].astype(str)
    r["ref_arxiv_id"] = [_arxiv_id(x) or _text(x) for x in r["ref_arxiv_id"]]
    dup = r.duplicated(["arxiv_id", "ref_arxiv_id"])
    if dup.any():
        print(f"[22] clean_refs: {int(dup.sum())} duplicate (arxiv_id, ref_arxiv_id) dropped")
        r = r[~dup].reset_index(drop=True)
    if r["ref_position"].isna().all():
        r["ref_position"] = r.groupby("arxiv_id", sort=False).cumcount()
    r["ref_title_clean"] = [H12["norm_title"](_text(t)) for t in r["ref_title"]]
    r["ref_tokens"] = [title_tokens(t) for t in r["ref_title_clean"]]
    r["ref_surnames"] = [author_surnames(a) for a in r["ref_authors"]]
    r["ref_first_surname"] = [s[0] if s else "" for s in r["ref_surnames"]]
    r["ref_n_authors"] = [len(s) for s in r["ref_surnames"]]
    years = [_year(y) for y in r["ref_year"]]
    r["ref_year"] = pd.array([y if y is not pd.NA else _year(_text(d)[:4] or None)
                              for y, d in zip(years, r["ref_submission_date"])], dtype="Int64")
    r["ref_doi_norm"] = [H12["norm_doi"](_text(d)) for d in r["ref_doi"]]
    r["ref_venue_clean"] = [H12["norm_title"](_text(v)) for v in r["ref_venue"]]
    r["ref_text"] = [" ".join(filter(None, (t, " ".join(s))))
                     for t, s in zip(r["ref_title_clean"], r["ref_surnames"])]
    return r


def cleaning_report(entries_c, refs_c):
    """Share of entries / references with each field after cleaning, by bib_source (README, poster)."""
    e = entries_c.assign(
        has_title=entries_c["title_clean"] != "", has_authors=entries_c["n_authors"] > 0,
        has_year=entries_c["year"].notna(), has_arxiv_hint=entries_c["arxiv_hint"] != "",
        has_doi=entries_c["doi_norm"] != "", arxiv_venue=entries_c["is_arxiv_venue"].astype(bool),
        has_context=entries_c["context_clean"] != "")
    shares = ["has_title", "has_authors", "has_year", "has_arxiv_hint", "has_doi", "arxiv_venue", "has_context"]
    rows = []
    for source, group in [("all entries", e), *e.groupby("bib_source")]:
        rows.append({"side": "entry", "group": source, "n": len(group),
                     **{c: round(float(group[c].mean()), 4) for c in shares}})
    r = refs_c
    rows.append({"side": "reference", "group": "all references", "n": len(r),
                 "has_title": round(float((r["ref_title_clean"] != "").mean()), 4),
                 "has_authors": round(float((r["ref_n_authors"] > 0).mean()), 4),
                 "has_year": round(float(r["ref_year"].notna().mean()), 4),
                 "has_doi": round(float((r["ref_doi_norm"] != "").mean()), 4)})
    return pd.DataFrame(rows)

## 2. Candidate pairs and labels

- `build_pairs` takes the Cartesian product **inside each publication** (the statement keeps pairs within
  one paper), paper by paper, and prints `Σ m·n`. `papers=` restricts it to the labelled publications.
- Labels come from `load_labels()` of `21_labeling` when it is loaded, else from the files of the contract
  (`read_label_files`). A manual label wins over an automatic one for the same entry.
- `attach_labels` keeps **only labelled entries** (the statement: an unlabelled entry has no row), adds
  `label` (0/1), `source` (`manual` / `auto` / `dev`) and `partition` (from `split.json`; every other
  labelled publication is `train`). It stops with the list of offending rows when a labelled reference is
  not among the references of its paper, or when an entry has two different labels from one source.

In [ ]:
def build_pairs(entries, refs, papers=None, chunk=500, verbose=True):
    """One row per (entry, reference) of the same paper: arxiv_id, citation_key, ref_arxiv_id, n_candidates."""
    e = entries[KEY]
    r = refs[["arxiv_id", "ref_arxiv_id"]]
    if papers is not None:
        papers = set(map(str, papers))
        e, r = e[e["arxiv_id"].isin(papers)], r[r["arxiv_id"].isin(papers)]
    n_refs = r.groupby("arxiv_id").size()
    ids = sorted(set(e["arxiv_id"]) & set(n_refs.index))
    frames = []
    for start in range(0, len(ids), chunk):
        part = set(ids[start:start + chunk])
        frames.append(e[e["arxiv_id"].isin(part)].merge(r[r["arxiv_id"].isin(part)], on="arxiv_id", how="inner"))
    pairs = (pd.concat(frames, ignore_index=True) if frames
             else pd.DataFrame(columns=CELL))
    pairs["n_candidates"] = pairs["arxiv_id"].map(n_refs).astype("int32")
    if verbose:
        m = e[e["arxiv_id"].isin(ids)].groupby("arxiv_id").size()
        print(f"[22] build_pairs: {len(ids)} papers with entries and references, "
              f"sum m*n = {int((m * n_refs.reindex(m.index)).sum()):,} pairs")
    return pairs


def read_label_files(label_dir=None):
    """Labels of the contract files -> arxiv_id, citation_key, ref_arxiv_id (None = no match), source."""
    label_dir = Path(label_dir or LABEL_DIR)
    frames = []
    for path in sorted((label_dir / "manual").glob("*.csv")):
        df = pd.read_csv(path, dtype=str, keep_default_na=False)
        frames.append(df[["arxiv_id", "citation_key", "ref_arxiv_id"]].assign(source="manual"))
    auto = label_dir / "auto" / "auto_labels.parquet"
    if auto.is_file():
        frames.append(pd.read_parquet(auto)[["arxiv_id", "citation_key", "ref_arxiv_id"]].assign(source="auto"))
    if not frames:
        raise FileNotFoundError(f"no labels under {label_dir}")
    labels = pd.concat(frames, ignore_index=True)
    labels["ref_arxiv_id"] = [(_arxiv_id(x) or _text(x).strip()) or None for x in labels["ref_arxiv_id"]]
    return labels


def read_split(path=None):
    """split.json -> {arxiv_id: 'test' | 'valid'}; any other labelled paper is 'train'."""
    path = Path(path or LABEL_DIR / "split.json")
    split = json.loads(path.read_text(encoding="utf-8"))
    out = {}
    for partition in ("test", "valid"):
        for ids in (split.get(partition) or {}).values():
            for arxiv_id in ([ids] if isinstance(ids, str) else ids):
                out[str(arxiv_id)] = partition
    return out


def _labels_one_per_entry(labels):
    """Manual before auto before dev; an entry with two different labels from one source is an error."""
    order = {"manual": 0, "auto": 1, "dev": 2}
    labels = labels.copy()
    labels["arxiv_id"] = labels["arxiv_id"].astype(str)
    labels["ref_arxiv_id"] = labels["ref_arxiv_id"].astype(object).where(labels["ref_arxiv_id"].notna(), None)
    labels["_rank"] = labels["source"].map(order).fillna(9)
    labels = labels.drop_duplicates(KEY + ["source", "ref_arxiv_id"])
    clash = labels[labels.duplicated(KEY + ["source"], keep=False)]
    if len(clash):
        raise ValueError(f"entries with two labels from one source:\n{clash.head(20).to_string()}")
    return labels.sort_values("_rank").drop_duplicates(KEY).drop(columns="_rank")


def attach_labels(pairs, labels, split=None, refs=None):
    """Pairs of labelled entries only, with label (0/1), source and partition."""
    labels = _labels_one_per_entry(labels)
    split = split or {}
    known = set(zip(pairs["arxiv_id"], pairs["ref_arxiv_id"]))
    if refs is not None:
        known |= set(zip(refs["arxiv_id"].astype(str), refs["ref_arxiv_id"]))
    positive = labels[labels["ref_arxiv_id"].notna()]
    missing = [(a, k, r) for a, k, r in positive[CELL].itertuples(index=False) if (a, r) not in known]
    if missing:
        raise ValueError(f"{len(missing)} labelled references are not references of their paper: {missing[:20]}")
    out = pairs.merge(labels.rename(columns={"ref_arxiv_id": "_gold"}), on=KEY, how="inner")
    out["label"] = (out["ref_arxiv_id"] == out["_gold"]).astype("int8")
    out["partition"] = [split.get(a, "train") for a in out["arxiv_id"]]
    lost = len(labels) - len(out[KEY].drop_duplicates())
    if lost:
        print(f"[22] attach_labels: {lost} labelled entries have no candidate (paper without references "
              f"or key not in entries); they score 0 in MRR")
    return out.drop(columns="_gold")

## 3. Features

Every feature is computed on the full `m × n` matrix of its paper and then read at the cells of `pairs`,
so the row and column ranks use **all** entries and references of the paper, labelled or not. A missing
value stays `NaN` (LightGBM handles it; `23_model` fills it for the logistic regression and adds a
`*_missing` flag). The table below is also the feature documentation of the README
(`feature_table_markdown()`); section 5 measures each one on the labelled pairs.

In [ ]:
FEATURE_DOC = {   # name: (group, formula, idea)
    "f_arxiv_eq": ("identifier", "`arxiv_hint == ref_arxiv_id`", "an entry that gives its arXiv id almost surely matches it"),
    "f_has_arxiv_hint": ("identifier", "the entry has an `arxiv_hint`", "tells the model when `f_arxiv_eq = 0` is real evidence"),
    "f_doi_eq": ("identifier", "`doi_norm` equals the reference DOI or `10.48550/arxiv.<ref id>`", "second identifier"),
    "f_title_tfidf_char": ("title", "cosine of char_wb 3-5-gram TF-IDF of the cleaned titles", "robust to typos, leftover LaTeX, hyphens"),
    "f_title_jaccard": ("title", "Jaccard of title content tokens", "shared content words"),
    "f_title_ratio": ("title", "`fuzz.ratio / 100`", "similarity of the whole string"),
    "f_title_partial": ("title", "`fuzz.partial_ratio / 100`", "truncated titles, with or without subtitle"),
    "f_title_len_diff": ("title", "`|len(t) - len(t_ref)| / max(len)`", "very different lengths mean different papers"),
    "f_first_surname_eq": ("author", "same first-author key", "how people remember a paper"),
    "f_surname_jaccard": ("author", "Jaccard of author key sets", "robust to missing authors"),
    "f_author_cover": ("author", "share of the entry's author keys found in the reference", "entries cut author lists with et al."),
    "f_n_authors_diff": ("author", "`|n_authors - ref_n_authors|`", "very different author counts"),
    "f_year_diff": ("year_venue", "`|year - ref_year|` (NaN if one is missing)", "preprint and journal version differ by 0-1 year"),
    "f_year_missing": ("year_venue", "one of the two years is missing", "lets the model ignore `f_year_diff`"),
    "f_venue_arxiv": ("year_venue", "the entry says it is an arXiv paper (journal, URL or DOI)", "arXiv-cited entries are the ones S2 lists with an id"),
    "f_venue_sim": ("year_venue", "`fuzz.token_set_ratio(venue, ref_venue) / 100` (NaN if one is empty)", "same journal or conference"),
    "f_raw_tfidf": ("fallback", "cosine of word TF-IDF of `raw_text` and reference title + author keys", "fallback for `\\bibitem` entries without a separate title"),
    "f_title_rank_in_row": ("context", "rank of `f_title_tfidf_char` within the entry's row (1 = best)", "compares this candidate with the other ones"),
    "f_gap_to_best": ("context", "row maximum of `f_title_tfidf_char` minus this cell", "distance to the best candidate"),
    "f_margin": ("context", "best minus second best `f_title_tfidf_char` in the row", "does the row have a clear winner"),
    "f_col_rank": ("context", "rank of the cell within the reference's column", "best match in both directions"),
    "f_n_candidates": ("context", "`n`, references of the paper", "a small `n` is an easier problem"),
    "f_ctx_tfidf": ("hierarchy", "cosine of word TF-IDF of the citing sentences (M2 `elements`) and the reference title", "the sentence that cites a paper often names its topic"),
    "f_n_cites": ("hierarchy", "times the entry is cited in the text", "much-cited entries are the main related work"),
    "f_pos_diff": ("experimental", "`|position/m - ref_position/n|`", "experimental: bibliography order vs the order of the S2 list"),
}
FEATURES = list(FEATURE_DOC)
FEATURE_GROUPS = {g: [f for f, d in FEATURE_DOC.items() if d[0] == g] for g in dict.fromkeys(d[0] for d in FEATURE_DOC.values())}
LOWER_IS_BETTER = {"f_title_len_diff", "f_n_authors_diff", "f_year_diff", "f_title_rank_in_row",
                   "f_gap_to_best", "f_col_rank", "f_pos_diff", "f_year_missing", "f_n_candidates"}
TEXT_MIN_DF = 2


def feature_table_markdown():
    """Feature documentation of the README, generated from FEATURE_DOC."""
    lines = ["| Feature | Group | Formula | Idea |", "|---|---|---|---|"]
    cell = lambda text: text.replace("|", "\|")   # a | inside a table cell, even in code, ends the cell
    lines += [f"| `{name}` | {g} | {cell(formula)} | {cell(idea)} |" for name, (g, formula, idea) in FEATURE_DOC.items()]
    return "\n".join(lines)


def _tfidf(min_df=TEXT_MIN_DF, **kwargs):
    return TfidfVectorizer(min_df=min_df, sublinear_tf=True, dtype=np.float32, **kwargs)


def _fit(vectorizer_factory, corpus):
    """Fit on the non-empty texts; min_df falls back to 1 on a tiny corpus. None if no vocabulary."""
    corpus = [t for t in corpus if t]
    for min_df in (TEXT_MIN_DF, 1):
        try:
            return vectorizer_factory(min_df).fit(corpus)
        except ValueError:   # empty vocabulary
            continue
    return None


def fit_text_models(entries_c, refs_c):
    """Vectorizers fitted on the texts of the corpus itself (no labels): char title, word raw, word context."""
    titles = list(entries_c["title_clean"]) + list(refs_c["ref_title_clean"])
    return {
        "title_char": _fit(lambda d: _tfidf(d, analyzer="char_wb", ngram_range=(3, 5)), titles),
        "raw_word": _fit(lambda d: _tfidf(d, analyzer="word", token_pattern=r"[a-z0-9]+", stop_words=list(STOP_WORDS)),
                         list(entries_c["raw_text"]) + list(refs_c["ref_text"])),
        "ctx_word": _fit(lambda d: _tfidf(d, analyzer="word", token_pattern=r"[a-z0-9]+", stop_words=list(STOP_WORDS)),
                         list(entries_c["context_clean"]) + list(refs_c["ref_title_clean"])),
        "bag": CountVectorizer(analyzer=lambda tokens: tokens, binary=True, dtype=np.float32).fit(
            list(entries_c["title_tokens"]) + list(refs_c["ref_tokens"])
            + list(entries_c["surnames"]) + list(refs_c["ref_surnames"]) + [["<empty>"]]),   # never an empty vocabulary
    }


def _cosine(vectorizer, left, right):
    """m x n cosine of two lists of texts (rows are L2-normalized by TF-IDF); NaN where a text is empty."""
    out = np.full((len(left), len(right)), np.nan, dtype=np.float32)
    if vectorizer is None:
        return out
    a, b = vectorizer.transform(left), vectorizer.transform(right)
    sim = (a @ b.T).toarray()
    ok = np.outer(np.array([bool(t) for t in left]), np.array([bool(t) for t in right]))
    out[ok] = sim[ok]
    return out


def _set_overlap(bag, left, right):
    """(intersection, |left|, |right|) matrices of token sets, through a binary bag of words."""
    a, b = bag.transform(left), bag.transform(right)
    inter = (a @ b.T).toarray()
    na = np.asarray(a.sum(axis=1)).ravel()
    nb = np.asarray(b.sum(axis=1)).ravel()
    return inter, na[:, None] * np.ones((1, len(right))), np.ones((len(left), 1)) * nb[None, :]


def _ratio(scorer, left, right):
    out = process.cdist(left, right, scorer=scorer, dtype=np.float32, workers=1) / 100.0
    ok = np.outer(np.array([bool(t) for t in left]), np.array([bool(t) for t in right]))
    return np.where(ok, out, np.nan).astype(np.float32)


def _paper_matrices(e, r, models):
    """{feature: m x n array} for one paper (e: its cleaned entries, r: its cleaned references)."""
    m, n = len(e), len(r)
    t_e, t_r = list(e["title_clean"]), list(r["ref_title_clean"])
    has_t = np.outer(np.array([bool(t) for t in t_e]), np.array([bool(t) for t in t_r]))
    f = {}
    hint = e["arxiv_hint"].to_numpy(dtype=object)[:, None]
    f["f_arxiv_eq"] = ((hint == r["ref_arxiv_id"].to_numpy(dtype=object)[None, :]) & (hint != "")).astype(np.float32)
    f["f_has_arxiv_hint"] = np.repeat((hint != "").astype(np.float32), n, axis=1)
    doi = e["doi_norm"].to_numpy(dtype=object)[:, None]
    ref_doi = r["ref_doi_norm"].to_numpy(dtype=object)[None, :]
    ref_arxiv_doi = np.array(["10.48550/arxiv." + x.lower() for x in r["ref_arxiv_id"]], dtype=object)[None, :]
    f["f_doi_eq"] = ((doi != "") & ((doi == ref_doi) | (doi == ref_arxiv_doi))).astype(np.float32)

    f["f_title_tfidf_char"] = _cosine(models["title_char"], t_e, t_r)
    inter, na, nb = _set_overlap(models["bag"], list(e["title_tokens"]), list(r["ref_tokens"]))
    union = na + nb - inter
    f["f_title_jaccard"] = np.where(has_t & (union > 0), inter / np.maximum(union, 1), np.nan).astype(np.float32)
    f["f_title_ratio"] = _ratio(fuzz.ratio, t_e, t_r)
    f["f_title_partial"] = _ratio(fuzz.partial_ratio, t_e, t_r)
    le = np.array([len(t) for t in t_e], dtype=np.float32)[:, None]
    lr = np.array([len(t) for t in t_r], dtype=np.float32)[None, :]
    f["f_title_len_diff"] = np.where(has_t, np.abs(le - lr) / np.maximum(np.maximum(le, lr), 1), np.nan).astype(np.float32)

    s_e, s_r = list(e["surnames"]), list(r["ref_surnames"])
    has_a = np.outer(np.array([bool(s) for s in s_e]), np.array([bool(s) for s in s_r]))
    first = e["first_surname"].to_numpy(dtype=object)[:, None]
    f["f_first_surname_eq"] = np.where(has_a, (first == r["ref_first_surname"].to_numpy(dtype=object)[None, :]), np.nan).astype(np.float32)
    inter, na, nb = _set_overlap(models["bag"], s_e, s_r)
    f["f_surname_jaccard"] = np.where(has_a, inter / np.maximum(na + nb - inter, 1), np.nan).astype(np.float32)
    f["f_author_cover"] = np.where(has_a, inter / np.maximum(na, 1), np.nan).astype(np.float32)
    f["f_n_authors_diff"] = np.where(has_a, np.abs(e["n_authors"].to_numpy()[:, None] - r["ref_n_authors"].to_numpy()[None, :]), np.nan).astype(np.float32)

    ye = e["year"].to_numpy(dtype="float64", na_value=np.nan)[:, None]
    yr = r["ref_year"].to_numpy(dtype="float64", na_value=np.nan)[None, :]
    f["f_year_diff"] = np.abs(ye - yr).astype(np.float32)
    f["f_year_missing"] = np.isnan(ye - yr).astype(np.float32)
    f["f_venue_arxiv"] = np.repeat(e["is_arxiv_venue"].to_numpy(dtype=np.float32)[:, None], n, axis=1)
    f["f_venue_sim"] = _ratio(fuzz.token_set_ratio, list(e["venue_clean"]), list(r["ref_venue_clean"]))
    f["f_raw_tfidf"] = _cosine(models["raw_word"], list(e["raw_text"]), list(r["ref_text"]))

    score = np.nan_to_num(f["f_title_tfidf_char"], nan=0.0)
    f["f_title_rank_in_row"] = rankdata(-score, method="min", axis=1).astype(np.float32)
    best = score.max(axis=1, keepdims=True)
    f["f_gap_to_best"] = (best - score).astype(np.float32)
    top2 = -np.sort(-score, axis=1)[:, :2]
    margin = top2[:, 0] - (top2[:, 1] if n > 1 else 0.0)
    f["f_margin"] = np.repeat(margin[:, None], n, axis=1).astype(np.float32)
    f["f_col_rank"] = rankdata(-score, method="min", axis=0).astype(np.float32)
    f["f_n_candidates"] = np.full((m, n), n, dtype=np.float32)

    f["f_ctx_tfidf"] = _cosine(models["ctx_word"], list(e["context_clean"]), t_r)
    f["f_n_cites"] = np.repeat(pd.to_numeric(e["n_cites"], errors="coerce").to_numpy(dtype=np.float32)[:, None], n, axis=1)
    pe = pd.to_numeric(e["position"], errors="coerce").to_numpy(dtype=np.float32)[:, None] / max(m, 1)
    rp = pd.to_numeric(r["ref_position"], errors="coerce").to_numpy(dtype=np.float32)
    pr = rp[None, :] / max(np.nanmax(rp) + 1 if len(rp) else 1, 1)
    f["f_pos_diff"] = np.abs(pe - pr).astype(np.float32)
    return f


def add_features(pairs, entries_c, refs_c, contexts=None, models=None, verbose=True):
    """pairs + one float32 column per name of FEATURES. contexts: optional {(arxiv_id, key): text} override."""
    t0 = time.time()
    if contexts is not None:
        entries_c = entries_c.copy()
        override = [contexts.get(k) for k in zip(entries_c["arxiv_id"], entries_c["citation_key"])]
        entries_c["context_clean"] = [_context_text(o) if o is not None else c
                                      for o, c in zip(override, entries_c["context_clean"])]
    models = models or fit_text_models(entries_c, refs_c)
    out = pairs.reset_index(drop=True).copy()
    values = {name: np.full(len(out), np.nan, dtype=np.float32) for name in FEATURES}
    e_by_paper = entries_c.groupby("arxiv_id", sort=False).indices
    r_by_paper = refs_c.groupby("arxiv_id", sort=False).indices
    for arxiv_id, cells in out.groupby("arxiv_id", sort=False).indices.items():
        e = entries_c.iloc[e_by_paper[arxiv_id]]
        r = refs_c.iloc[r_by_paper[arxiv_id]]
        row = pd.Series(np.arange(len(e)), index=e["citation_key"].to_numpy())
        col = pd.Series(np.arange(len(r)), index=r["ref_arxiv_id"].to_numpy())
        i = row.reindex(out["citation_key"].to_numpy()[cells]).to_numpy()
        j = col.reindex(out["ref_arxiv_id"].to_numpy()[cells]).to_numpy()
        if np.isnan(i.astype(float)).any() or np.isnan(j.astype(float)).any():
            raise KeyError(f"{arxiv_id}: pairs with an entry or a reference that is not in the cleaned frames")
        i, j = i.astype(int), j.astype(int)
        for name, matrix in _paper_matrices(e, r, models).items():
            values[name][cells] = matrix[i, j]
    for name in FEATURES:
        out[name] = values[name]
    if verbose:
        print(f"[22] add_features: {len(out):,} pairs x {len(FEATURES)} features in {time.time() - t0:.1f}s")
    return out

## 4. Ranking and MRR@5

`rank_candidates` orders the references of each entry by a score (ties broken by `ref_arxiv_id`, so the
order never depends on the row order) and keeps the top `k`. `mrr_at_k` follows the statement:
`MRR = (1/|Q|) Σ 1/rank_i`, 0 when the true reference is not in the top 5.

- `nil_mode="zero"`: the statement's wording; an entry **without** a true reference counts in `|Q|` with 0.
- `nil_mode="skip"`: only entries with a true reference.

Both are reported until the teacher says which one is official. `23_model` reuses these functions.

In [ ]:
def rank_candidates(pairs, scores, k=5):
    """{(arxiv_id, citation_key): [ref_arxiv_id, ...]} best first, at most k (NaN scores rank last)."""
    df = pairs[CELL].copy()
    df["_s"] = np.nan_to_num(np.asarray(scores, dtype=np.float64), nan=-np.inf)
    df = df.sort_values(KEY + ["_s", "ref_arxiv_id"], ascending=[True, True, False, True], kind="mergesort")
    top = df.groupby(KEY, sort=False).head(k)
    return {key: list(group["ref_arxiv_id"]) for key, group in top.groupby(KEY, sort=False)}


def gold_of(labeled_pairs):
    """{(arxiv_id, citation_key): true ref_arxiv_id or None} from labelled pairs."""
    gold = {key: None for key in zip(labeled_pairs["arxiv_id"], labeled_pairs["citation_key"])}
    pos = labeled_pairs[labeled_pairs["label"] == 1]
    gold.update({(a, c): r for a, c, r in pos[CELL].itertuples(index=False)})
    return gold


def mrr_at_k(ranked, gold, k=5, nil_mode="zero"):
    """Mean reciprocal rank over the entries of `gold` (see above); an entry missing from `ranked` scores 0."""
    if nil_mode not in ("zero", "skip"):
        raise ValueError("nil_mode is 'zero' or 'skip'")
    total, count = 0.0, 0
    for key, true_ref in gold.items():
        if true_ref is None:
            if nil_mode == "zero":
                count += 1
            continue
        count += 1
        candidates = ranked.get(key, [])[:k]
        if true_ref in candidates:
            total += 1.0 / (candidates.index(true_ref) + 1)
    return total / count if count else float("nan")


def hit_at_k(ranked, gold, k=1):
    """Share of entries with a true reference whose reference is in the top k."""
    keys = [key for key, ref in gold.items() if ref is not None]
    return float(np.mean([gold[key] in ranked.get(key, [])[:k] for key in keys])) if keys else float("nan")

## 5. Feature analysis

The statement asks to justify every feature with data. For each feature on the labelled pairs,
`feature_report` gives the mean on positive and negative pairs, the **AUC of the feature alone**
(`auc_oriented` = max(AUC, 1 − AUC), with the direction in which it separates) and the **MRR@5 of ranking
by that feature alone** (oriented the same way). A `row_constant` feature (`f_has_arxiv_hint`, `f_margin`,
`f_n_cites`, ...) has the same value for every candidate of an entry: it cannot rank on its own (its MRR is
the tie-break order), it only tells the model how much to trust the other features. `plot_feature_histograms` draws positive vs negative
histograms per group (log y axis, since negatives outnumber positives by about `n` to 1);
`feature_findings` gives the numbers quoted in the README and on the poster.

In [ ]:
def feature_report(labeled_pairs, features=None, k=5):
    """One row per feature: coverage, mean on positives/negatives, AUC, single-feature MRR@k (both nil modes)."""
    features = features or FEATURES
    y = labeled_pairs["label"].to_numpy()
    gold = gold_of(labeled_pairs)
    rows = []
    for name in features:
        x = labeled_pairs[name].to_numpy(dtype=np.float64)
        ok = ~np.isnan(x)
        auc = (roc_auc_score(y[ok], x[ok]) if ok.any() and len(np.unique(y[ok])) == 2
               and len(np.unique(x[ok])) > 1 else np.nan)
        sign = -1.0 if (name in LOWER_IS_BETTER if np.isnan(auc) else auc < 0.5) else 1.0
        ranked = rank_candidates(labeled_pairs, sign * x, k=k)
        rows.append({
            "feature": name, "group": FEATURE_DOC[name][0], "coverage": round(float(ok.mean()), 4),
            "row_constant": bool(labeled_pairs.groupby(KEY)[name].nunique(dropna=False).max() <= 1),
            "mean_pos": float(np.nanmean(x[y == 1])) if (ok & (y == 1)).any() else np.nan,
            "mean_neg": float(np.nanmean(x[y == 0])) if (ok & (y == 0)).any() else np.nan,
            "auc": auc, "direction": "higher" if sign > 0 else "lower",
            "auc_oriented": max(auc, 1 - auc) if not np.isnan(auc) else np.nan,
            "mrr5_zero": mrr_at_k(ranked, gold, k, "zero"), "mrr5_skip": mrr_at_k(ranked, gold, k, "skip"),
        })
    return pd.DataFrame(rows).round(4)


def plot_feature_histograms(labeled_pairs, out_dir=None, bins=30):
    """feat_<group>.png: positive vs negative histogram of every feature of the group, log y axis."""
    import matplotlib.pyplot as plt
    out_dir = Path(out_dir or M3_FIG_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    pos = labeled_pairs["label"] == 1
    paths = []
    for group, names in FEATURE_GROUPS.items():
        cols = min(4, len(names))
        rows = -(-len(names) // cols)
        fig, axes = plt.subplots(rows, cols, figsize=(3.6 * cols, 2.8 * rows), squeeze=False)
        for ax, name in zip(axes.ravel(), names):
            x = labeled_pairs[name].astype(float)
            values = x.dropna()
            ax.set_title(name, fontsize=9)
            if values.empty:
                ax.text(0.5, 0.5, "no value in these pairs", ha="center", va="center", fontsize=8)
                continue
            edges = np.histogram_bin_edges(values, bins=bins) if len(values) and values.nunique() > 1 else bins
            ax.hist(x[~pos].dropna(), bins=edges, alpha=0.6, label="negative", color="#8c8c8c")
            ax.hist(x[pos].dropna(), bins=edges, alpha=0.7, label="positive", color="#1f77b4")
            ax.set_yscale("log")
        for ax in axes.ravel()[len(names):]:
            ax.axis("off")
        for ax in axes.ravel():
            if ax.get_legend_handles_labels()[0]:
                ax.legend(fontsize=8)
                break
        fig.suptitle(f"Features: {group} (positive vs negative pairs)", fontsize=10)
        fig.tight_layout()
        path = out_dir / f"feat_{group}.png"
        fig.savefig(path, dpi=120)
        plt.close(fig)
        paths.append(path)
    return paths


def feature_findings(labeled_pairs, entries_c, report=None):
    """Numbers behind the README / poster sentences on the features."""
    report = report if report is not None else feature_report(labeled_pairs)
    auc = report.set_index("feature")["auc_oriented"]
    pos = labeled_pairs[labeled_pairs["label"] == 1]
    labelled_keys = labeled_pairs[KEY].drop_duplicates()
    e = entries_c.merge(labelled_keys, on=KEY)
    no_title = e[(e["title_clean"] == "")]
    no_title_pos = pos.merge(no_title[KEY], on=KEY)
    year = pos["f_year_diff"].dropna()
    return {
        "entries_labelled": len(e),
        "entries_with_true_ref": int(len(pos)),
        "share_entries_with_arxiv_hint": round(float((e["arxiv_hint"] != "").mean()), 4) if len(e) else None,
        "share_true_pairs_with_arxiv_eq": round(float(pos["f_arxiv_eq"].mean()), 4) if len(pos) else None,
        "true_pairs_title_ratio_median": round(float(pos["f_title_ratio"].median()), 4) if len(pos) else None,
        "true_pairs_title_ratio_share_ge_0.9": round(float((pos["f_title_ratio"] >= 0.9).mean()), 4) if len(pos) else None,
        "true_pairs_year_diff": {str(int(k)): int(v) for k, v in year.clip(upper=3).value_counts().sort_index().items()},
        "entries_without_title": {str(k): int(v) for k, v in no_title.groupby("bib_source").size().items()},
        "raw_tfidf_on_true_pairs_without_title": round(float(no_title_pos["f_raw_tfidf"].mean()), 4) if len(no_title_pos) else None,
        "auc_f_pos_diff": None if pd.isna(auc.get("f_pos_diff")) else float(auc["f_pos_diff"]),
        "keep_f_pos_diff": bool(auc.get("f_pos_diff", 0.5) >= 0.55),
    }

## 6. Full run (`25_run_m3`)

`run_features()` reads the contract files, cleans both sides, builds the pairs of the labelled
publications, attaches labels and partitions, computes the features and writes `data/m3/pairs.parquet`,
the cleaning and feature reports and the figures. Any input can be passed as a DataFrame instead.

In [ ]:
def run_features(entries=None, refs=None, labels=None, split=None, save=True, figures=True):
    """End to end on the real data; returns (pairs with features, feature report, cleaning report)."""
    t0 = time.time()
    entries = entries if entries is not None else pd.read_parquet(M3_DIR / "entries.parquet")
    refs = refs if refs is not None else pd.read_parquet(M3_DIR / "refs.parquet")
    if labels is None:
        labels = globals()["load_labels"]() if "load_labels" in globals() else read_label_files()
    split = split if split is not None else read_split()
    entries_c, refs_c = clean_entries(entries), clean_refs(refs)
    cleaning = cleaning_report(entries_c, refs_c)
    labelled_papers = sorted(set(labels["arxiv_id"].astype(str)))
    pairs = build_pairs(entries_c, refs_c, papers=labelled_papers)
    pairs = attach_labels(pairs, labels, split, refs=refs_c)
    models = fit_text_models(entries_c, refs_c)     # whole corpus: more n-gram statistics, still no labels
    pairs = add_features(pairs, entries_c, refs_c, models=models)
    report = feature_report(pairs)
    if save:
        M3_DIR.mkdir(parents=True, exist_ok=True)
        out = pairs.copy()
        for column in ("arxiv_id", "source", "partition"):
            out[column] = out[column].astype("category")
        out.to_parquet(PAIRS_PATH, index=False)
        report.to_csv(FEATURE_REPORT_PATH, index=False)
        cleaning.to_csv(CLEANING_REPORT_PATH, index=False)
        print(f"[22] wrote {PAIRS_PATH} ({len(pairs):,} rows), {FEATURE_REPORT_PATH}, {CLEANING_REPORT_PATH}")
    if figures:
        plot_feature_histograms(pairs)
    print(f"[22] run_features done in {time.time() - t0:.1f}s | partitions: "
          f"{pairs.drop_duplicates(KEY)['partition'].value_counts().to_dict()}")
    return pairs, report, cleaning

## 7. Dev sample (before the real labels)

The 31-paper sample of `tools/fetch_bib_sample.py` (`data/parsed/bibtex_sample/bibtex.parquet`) and the
S2 references in `data/raw/<id>/references.json` give the two frames of the contract (without citation
contexts). `dev_labels` is a **weak** labelling used only to try the code: an entry is positive for the
reference with the same arXiv id or exactly the same cleaned title, otherwise it has no match. Its
numbers are never reported as results (they favour `f_arxiv_eq` by construction).

In [ ]:
DEV_BIBTEX = DATA_DIR.parent / "parsed" / "bibtex_sample" / "bibtex.parquet"


def dev_frames(bibtex_path=DEV_BIBTEX, raw_dir=None):
    """(entries, refs) in the contract shape from the local dev sample."""
    raw_dir = Path(raw_dir or DATA_DIR)
    entries = pd.read_parquet(bibtex_path)
    rows = []
    for arxiv_id in sorted(entries["arxiv_id"].unique()):
        path = raw_dir / H03["normalize_ref_arxiv_id"](arxiv_id)[1] / "references.json"
        doc = json.loads(path.read_text(encoding="utf-8")) if path.is_file() else {"references": []}
        for position, ref in enumerate(doc.get("references") or []):
            if ref.get("arxiv_id"):
                rows.append({"arxiv_id": arxiv_id, "ref_arxiv_id": ref["arxiv_id"], "ref_title": ref.get("title"),
                             "ref_authors": ref.get("authors") or [], "ref_submission_date": ref.get("submission_date"),
                             "ref_s2_id": ref.get("s2_id"), "ref_year": ref.get("year"),
                             "ref_doi": (ref.get("external_ids") or {}).get("DOI"), "ref_venue": ref.get("venue"),
                             "ref_position": position})
    return entries, pd.DataFrame(rows)


def dev_labels(entries_c, refs_c):
    """Weak dev labels: same arXiv id, else same cleaned title (unique in the paper), else no match."""
    rows = []
    refs_by_paper = {a: g for a, g in refs_c.groupby("arxiv_id")}
    for e in entries_c.itertuples(index=False):
        r = refs_by_paper.get(e.arxiv_id)
        if r is None:
            continue
        match = None
        if e.arxiv_hint and (r["ref_arxiv_id"] == e.arxiv_hint).any():
            match = e.arxiv_hint
        elif e.title_clean:
            same = r[r["ref_title_clean"] == e.title_clean]
            match = same["ref_arxiv_id"].iloc[0] if len(same) == 1 else None
        rows.append({"arxiv_id": e.arxiv_id, "citation_key": e.citation_key, "ref_arxiv_id": match, "source": "dev"})
    return pd.DataFrame(rows)

## 8. Tests

Offline tests on hand-made rows (cleaning, pairs, features, labels, MRR), then the whole chain on the dev
sample when it is present. `RUN_TEST = False` (as `23_model` and `25_run_m3` load this notebook) runs none
of it.

In [ ]:
import unittest


def _entry(arxiv_id="2601.00001", key="k", title="", authors=(), year=None, doi=None, journal=None, url=None, **extra):
    return {"arxiv_id": arxiv_id, "citation_key": key, "entry_type": "article", "title": title,
            "authors": list(authors), "year": year, "doi": doi, "journal": journal, "url": url, **extra}


def _ref(arxiv_id="2601.00001", ref="2301.00001", title="", authors=(), year=None, date=None, doi=None, venue=None):
    return {"arxiv_id": arxiv_id, "ref_arxiv_id": ref, "ref_title": title, "ref_authors": list(authors),
            "ref_submission_date": date, "ref_s2_id": None, "ref_year": year, "ref_doi": doi, "ref_venue": venue}


class FeatureTests(unittest.TestCase):
    # ---------- cleaning ----------
    def test_title_latex_accents_and_fonts(self):
        e = clean_entries(pd.DataFrame([_entry(title=r"{\em Deep} Learning in M{\"u}nster: The {GPT}-4 Case")]))
        self.assertEqual(e.loc[0, "title_clean"], "deep learning in munster the gpt 4 case")
        self.assertEqual(e.loc[0, "title_tokens"], ["deep", "learning", "munster", "gpt", "4", "case"])

    def test_surname_last_first_and_initials(self):
        self.assertEqual(author_surname("Smith, John"), "smith")
        self.assertEqual(author_surname("A. B. Smith"), "smith")
        self.assertEqual(author_surname("Lambert M{\\\"u}nster"), "munster")

    def test_surname_particles_hyphens_suffixes(self):
        for name in ("Johannes Diderik van der Waals", "van der Waals, J. D.", "J. D. van der Waals"):
            self.assertEqual(author_surname(name), "waals", name)
        self.assertEqual(author_surname("Martin Luther King Jr."), "king")
        self.assertEqual(author_surname("King, Jr., Martin Luther"), "king")
        self.assertEqual(author_surname("Pierre-Gilles de Gennes"), "gennes")
        self.assertEqual(author_surname("Jean-Pierre Serre-Martin"), "serremartin")

    def test_et_al_and_others_dropped(self):
        self.assertEqual(author_surnames(["A. Smith", "others"]), ["smith"])
        self.assertEqual(author_surnames(["A. Smith", "B. Jones et al."]), ["smith", "jones"])
        self.assertEqual(author_surnames(["et al."]), [])
        self.assertEqual(author_surnames(None), [])

    def test_arxiv_hint_old_style_url(self):
        e = clean_entries(pd.DataFrame([_entry(url="https://arxiv.org/abs/hep-th/9901001v2")]))
        self.assertEqual(e.loc[0, "arxiv_hint"], "hep-th/9901001")
        self.assertTrue(e.loc[0, "is_arxiv_venue"])

    def test_arxiv_hint_from_doi_and_journal(self):
        e = clean_entries(pd.DataFrame([_entry(key="a", doi="10.48550/arXiv.2310.06825"),
                                        _entry(key="b", journal="arXiv preprint arXiv:2210.06591")]))
        self.assertEqual(list(e["arxiv_hint"]), ["2310.06825", "2210.06591"])
        self.assertEqual(e.loc[0, "doi_norm"], "10.48550/arxiv.2310.06825")
        self.assertTrue(e["is_arxiv_venue"].all())

    def test_missing_year(self):
        e = clean_entries(pd.DataFrame([_entry(title="A b c", year=None)]))
        r = clean_refs(pd.DataFrame([_ref(title="A b c", year=2020)]))
        p = add_features(build_pairs(e, r, verbose=False), e, r, verbose=False)
        self.assertTrue(pd.isna(e.loc[0, "year"]))
        self.assertEqual(p.loc[0, "f_year_missing"], 1.0)
        self.assertTrue(np.isnan(p.loc[0, "f_year_diff"]))

    def test_ref_year_from_submission_date(self):
        r = clean_refs(pd.DataFrame([_ref(year=None, date="2019-07-02"), _ref(ref="2301.00002", year=2021, date="2019-01-01")]))
        self.assertEqual(list(r["ref_year"]), [2019, 2021])

    def test_bibitem_without_title_uses_raw_text(self):
        e = clean_entries(pd.DataFrame([_entry(title=None, authors=["A. Vaswani", "N. Shazeer"], journal="NeurIPS",
                                               bib_source="bibitem")]))
        r = clean_refs(pd.DataFrame([_ref(title="Attention is all you need", authors=["Ashish Vaswani", "Noam Shazeer"]),
                                     _ref(ref="2301.00002", title="Other paper", authors=["X. Li"])]))
        p = add_features(build_pairs(e, r, verbose=False), e, r, verbose=False)
        self.assertEqual(e.loc[0, "title_clean"], "")
        self.assertEqual(e.loc[0, "raw_text"], "vaswani shazeer neurips")
        self.assertTrue(p["f_title_ratio"].isna().all())
        self.assertGreater(p.loc[0, "f_raw_tfidf"], p.loc[1, "f_raw_tfidf"])

    def test_clean_refs_drops_duplicates(self):
        r = clean_refs(pd.DataFrame([_ref(ref="2301.00001v2"), _ref(ref="2301.00001")]))
        self.assertEqual(list(r["ref_arxiv_id"]), ["2301.00001"])

    # ---------- pairs and labels ----------
    def test_build_pairs_is_m_times_n_per_paper(self):
        e = clean_entries(pd.DataFrame([_entry("p1", "a"), _entry("p1", "b"), _entry("p1", "c"), _entry("p2", "x")]))
        r = clean_refs(pd.DataFrame([_ref("p1", "1"), _ref("p1", "2"), _ref("p2", "3"), _ref("p3", "4")]))
        p = build_pairs(e, r, verbose=False)
        self.assertEqual(len(p), 3 * 2 + 1 * 1)
        self.assertEqual(set(p[p["arxiv_id"] == "p2"]["ref_arxiv_id"]), {"3"})
        self.assertEqual(p.groupby("arxiv_id")["n_candidates"].first().to_dict(), {"p1": 2, "p2": 1})

    def test_attach_labels_manual_wins_and_partitions(self):
        e = clean_entries(pd.DataFrame([_entry("p1", "a"), _entry("p1", "b"), _entry("p1", "c"), _entry("p2", "x")]))
        r = clean_refs(pd.DataFrame([_ref("p1", "2301.00001"), _ref("p1", "2301.00002"), _ref("p2", "2301.00003")]))
        labels = pd.DataFrame([
            {"arxiv_id": "p1", "citation_key": "a", "ref_arxiv_id": "2301.00001", "source": "auto"},
            {"arxiv_id": "p1", "citation_key": "a", "ref_arxiv_id": "2301.00002", "source": "manual"},
            {"arxiv_id": "p1", "citation_key": "b", "ref_arxiv_id": None, "source": "manual"},
            {"arxiv_id": "p2", "citation_key": "x", "ref_arxiv_id": "2301.00003", "source": "auto"}])
        p = attach_labels(build_pairs(e, r, verbose=False), labels, {"p2": "test"})
        self.assertNotIn("c", set(p["citation_key"]))                      # unlabelled entry: no row
        a = p[p["citation_key"] == "a"].set_index("ref_arxiv_id")
        self.assertEqual(a["label"].to_dict(), {"2301.00001": 0, "2301.00002": 1})
        self.assertEqual(set(a["source"]), {"manual"})
        self.assertEqual(p[p["citation_key"] == "b"]["label"].sum(), 0)
        self.assertEqual(p.groupby("arxiv_id")["partition"].first().to_dict(), {"p1": "train", "p2": "test"})
        self.assertTrue((p.groupby(KEY)["label"].sum() <= 1).all())

    def test_attach_labels_rejects_foreign_reference(self):
        e = clean_entries(pd.DataFrame([_entry("p1", "a")]))
        r = clean_refs(pd.DataFrame([_ref("p1", "2301.00001")]))
        bad = pd.DataFrame([{"arxiv_id": "p1", "citation_key": "a", "ref_arxiv_id": "2399.99999", "source": "manual"}])
        with self.assertRaises(ValueError):
            attach_labels(build_pairs(e, r, verbose=False), bad)

    # ---------- features ----------
    def test_identical_titles(self):
        title = "Attention Is All You Need"
        e = clean_entries(pd.DataFrame([_entry(title=title, authors=["Ashish Vaswani"])]))
        r = clean_refs(pd.DataFrame([_ref(title=title, authors=["A. Vaswani"]),
                                     _ref(ref="2301.00002", title="Convolutional sequence to sequence learning")]))
        p = add_features(build_pairs(e, r, verbose=False), e, r, verbose=False).set_index("ref_arxiv_id")
        self.assertAlmostEqual(float(p.loc["2301.00001", "f_title_tfidf_char"]), 1.0, places=5)
        self.assertEqual(p.loc["2301.00001", "f_title_ratio"], 1.0)
        self.assertEqual(p.loc["2301.00001", "f_title_jaccard"], 1.0)
        self.assertEqual(p.loc["2301.00001", "f_first_surname_eq"], 1.0)
        self.assertEqual(p.loc["2301.00001", "f_title_len_diff"], 0.0)

    def test_identifier_features(self):
        e = clean_entries(pd.DataFrame([_entry(key="a", url="https://arxiv.org/abs/2301.00001"),
                                        _entry(key="b", doi="10.1103/PhysRevE.58.2749"), _entry(key="c")]))
        r = clean_refs(pd.DataFrame([_ref(ref="2301.00001"), _ref(ref="2301.00002", doi="10.1103/physreve.58.2749")]))
        p = add_features(build_pairs(e, r, verbose=False), e, r, verbose=False).set_index(["citation_key", "ref_arxiv_id"])
        self.assertEqual(p["f_arxiv_eq"].sum(), 1.0)
        self.assertEqual(p.loc[("a", "2301.00001"), "f_arxiv_eq"], 1.0)
        self.assertEqual(p.loc[("a", "2301.00001"), "f_doi_eq"], 0.0)
        self.assertEqual(p.loc[("b", "2301.00002"), "f_doi_eq"], 1.0)
        self.assertEqual(p.loc[("c", "2301.00001"), "f_has_arxiv_hint"], 0.0)

    def test_row_and_column_ranks(self):
        e = clean_entries(pd.DataFrame([_entry(key="a", title="graph neural networks for molecules"),
                                        _entry(key="b", title="graph neural networks")]))
        r = clean_refs(pd.DataFrame([_ref(ref="1", title="graph neural networks for molecules"),
                                     _ref(ref="2", title="graph neural networks"),
                                     _ref(ref="3", title="quantum chromodynamics on the lattice")]))
        p = add_features(build_pairs(e, r, verbose=False), e, r, verbose=False).set_index(["citation_key", "ref_arxiv_id"])
        self.assertEqual(p.loc[("a", "1"), "f_title_rank_in_row"], 1.0)
        self.assertEqual(p.loc[("a", "3"), "f_title_rank_in_row"], 3.0)
        self.assertEqual(p.loc[("b", "2"), "f_title_rank_in_row"], 1.0)
        self.assertEqual(p.loc[("b", "2"), "f_col_rank"], 1.0)
        self.assertEqual(p.loc[("a", "1"), "f_gap_to_best"], 0.0)
        self.assertGreater(p.loc[("a", "3"), "f_gap_to_best"], 0.5)
        self.assertAlmostEqual(float(p.loc[("a", "1"), "f_margin"]), float(p.loc[("a", "3"), "f_margin"]))
        self.assertEqual(set(p["f_n_candidates"]), {3.0})

    def test_features_are_computed_on_the_full_paper(self):
        # The column rank must not change when only some entries are labelled.
        e = clean_entries(pd.DataFrame([_entry(key="a", title="graph neural networks"),
                                        _entry(key="b", title="graph neural networks for molecules")]))
        r = clean_refs(pd.DataFrame([_ref(ref="1", title="graph neural networks for molecules")]))
        full = add_features(build_pairs(e, r, verbose=False), e, r, verbose=False)
        part = add_features(build_pairs(e, r, verbose=False).iloc[[0]], e, r, verbose=False)
        self.assertEqual(part.loc[0, "f_col_rank"], full.loc[0, "f_col_rank"])
        self.assertEqual(part.loc[0, "f_col_rank"], 2.0)

    def test_context_feature(self):
        e = clean_entries(pd.DataFrame([_entry(key="a", context=[r"Transformers \cite{a} use attention."]),
                                        _entry(key="b")]))
        r = clean_refs(pd.DataFrame([_ref(ref="1", title="Attention is all you need"), _ref(ref="2", title="Lattice QCD")]))
        p = add_features(build_pairs(e, r, verbose=False), e, r, verbose=False).set_index(["citation_key", "ref_arxiv_id"])
        self.assertGreater(p.loc[("a", "1"), "f_ctx_tfidf"], p.loc[("a", "2"), "f_ctx_tfidf"])
        self.assertTrue(np.isnan(p.loc[("b", "1"), "f_ctx_tfidf"]))

    # ---------- metric and report ----------
    def test_mrr_hand_example(self):
        ranked = {("p", "e1"): ["r1", "x", "y"], ("p", "e2"): ["x", "y", "r2"],
                  ("p", "e3"): ["a", "b", "c", "d", "e", "r3"], ("p", "e4"): ["a", "b"]}
        gold = {("p", "e1"): "r1", ("p", "e2"): "r2", ("p", "e3"): "r3", ("p", "e4"): None}
        self.assertAlmostEqual(mrr_at_k(ranked, gold, 5, "zero"), (1 + 1 / 3) / 4, places=4)
        self.assertAlmostEqual(mrr_at_k(ranked, gold, 5, "skip"), (1 + 1 / 3) / 3, places=4)
        self.assertAlmostEqual(round(mrr_at_k(ranked, gold, 5, "zero"), 4), 0.3333)
        self.assertAlmostEqual(round(mrr_at_k(ranked, gold, 5, "skip"), 4), 0.4444)
        self.assertAlmostEqual(hit_at_k(ranked, gold, 1), 1 / 3)
        self.assertEqual(mrr_at_k({}, {("p", "e1"): "r1"}), 0.0)

    def test_rank_candidates_ties_and_nan(self):
        p = pd.DataFrame({"arxiv_id": ["p"] * 3, "citation_key": ["e"] * 3, "ref_arxiv_id": ["c", "a", "b"]})
        self.assertEqual(rank_candidates(p, [0.5, 0.5, np.nan])[("p", "e")], ["a", "c", "b"])

    def test_feature_report_on_toy(self):
        e = clean_entries(pd.DataFrame([_entry(key=f"k{i}", title=t, url=f"https://arxiv.org/abs/2301.0000{i}")
                                        for i, t in enumerate(["alpha beta gamma", "delta epsilon", "zeta eta theta"])]))
        r = clean_refs(pd.DataFrame([_ref(ref=f"2301.0000{i}", title=t)
                                     for i, t in enumerate(["alpha beta gamma", "delta epsilon", "zeta eta theta"])]))
        labels = pd.DataFrame({"arxiv_id": e["arxiv_id"], "citation_key": e["citation_key"],
                               "ref_arxiv_id": r["ref_arxiv_id"], "source": "manual"})
        p = add_features(attach_labels(build_pairs(e, r, verbose=False), labels), e, r, verbose=False)
        rep = feature_report(p).set_index("feature")
        self.assertEqual(rep.loc["f_arxiv_eq", "auc"], 1.0)
        self.assertEqual(rep.loc["f_arxiv_eq", "mrr5_zero"], 1.0)
        self.assertEqual(rep.loc["f_title_rank_in_row", "direction"], "lower")
        self.assertEqual(set(rep.index), set(FEATURES))


if RUN_TEST:
    _res = unittest.TextTestRunner(verbosity=1).run(unittest.defaultTestLoader.loadTestsFromTestCase(FeatureTests))
    assert _res.wasSuccessful(), "22 tests failed"

In [ ]:
# Whole chain on the dev sample (weak labels): checks that the code runs on real rows, not a result.
if RUN_TEST and DEV_BIBTEX.is_file():
    _t0 = time.time()
    dev_entries, dev_refs = dev_frames()
    dev_e, dev_r = clean_entries(dev_entries), clean_refs(dev_refs)
    display(cleaning_report(dev_e, dev_r))
    dev_lab = dev_labels(dev_e, dev_r)
    dev_pairs = add_features(attach_labels(build_pairs(dev_e, dev_r), dev_lab, refs=dev_r), dev_e, dev_r)
    print(f"dev: {dev_entries['arxiv_id'].nunique()} papers, {len(dev_lab)} labelled entries, "
          f"{int(dev_lab['ref_arxiv_id'].notna().sum())} with a match | {time.time() - _t0:.1f}s")
    dev_report = feature_report(dev_pairs)
    display(dev_report.sort_values("auc_oriented", ascending=False))
    print(json.dumps(feature_findings(dev_pairs, dev_e, dev_report), indent=1))
    print("figures:", [p.name for p in plot_feature_histograms(dev_pairs, LOG_DIR / "m3_dev" / "figures")])